## Assignment No.7

## Multiple Linear Regression

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('ToyotaCorolla-MLR.csv')

df.drop(columns=['Cylinders'], inplace=True)

df.loc[df['cc'] == 16000, 'cc'] = 1600
df = pd.get_dummies(df, columns=['Fuel_Type'], drop_first=True)
df[df.select_dtypes(include='bool').columns] = df.select_dtypes(include='bool').astype(int)

print(df.shape)           # (1436, 11)
print(df.isnull().sum())  # All zeros - no missing values
print(df.describe())

(1436, 11)
Price               0
Age_08_04           0
KM                  0
HP                  0
Automatic           0
cc                  0
Doors               0
Gears               0
Weight              0
Fuel_Type_Diesel    0
Fuel_Type_Petrol    0
dtype: int64
              Price    Age_08_04             KM           HP    Automatic  \
count   1436.000000  1436.000000    1436.000000  1436.000000  1436.000000   
mean   10730.824513    55.947075   68533.259749   101.502089     0.055710   
std     3626.964585    18.599988   37506.448872    14.981080     0.229441   
min     4350.000000     1.000000       1.000000    69.000000     0.000000   
25%     8450.000000    44.000000   43000.000000    90.000000     0.000000   
50%     9900.000000    61.000000   63389.500000   110.000000     0.000000   
75%    11950.000000    70.000000   87020.750000   110.000000     0.000000   
max    32500.000000    80.000000  243000.000000   192.000000     1.000000   

                cc        Doors        G

In [5]:
X = df.drop(columns=['Price'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")  # (1148, 10)
print(f"Testing set:  {X_test.shape}")   # (288, 10)

# Standardize for Ridge and Lasso (fit on train only to prevent data leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

Training set: (1148, 10)
Testing set:  (288, 10)


In [6]:
model1 = LinearRegression()
model1.fit(X_train, y_train)

coeff_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': model1.coef_})
print(coeff_df.sort_values('Coefficient', ascending=False))
print(f"Intercept: {model1.intercept_:.2f}")

            Feature  Coefficient
8  Fuel_Type_Diesel  2035.927064
9  Fuel_Type_Petrol  1289.973942
6             Gears   478.134018
3         Automatic   207.770928
2                HP    41.561829
7            Weight    25.005582
1                KM    -0.015451
4                cc    -3.136174
5             Doors   -25.626048
0         Age_08_04  -119.787684
Intercept: -11282.55


In [7]:
top_feats = ['Age_08_04', 'KM', 'HP', 'Weight', 'cc']
model2 = LinearRegression()
model2.fit(X_train[top_feats], y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [8]:
model3 = LinearRegression()
model3.fit(X_train_sc, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [10]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2   = r2_score(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    print(f"{name:35s} | R²={r2:.4f} | RMSE={rmse:.2f} | MAE={mae:.2f}")
    return r2, rmse, mae

evaluate_model("Model 1: Full MLR",         LinearRegression(), X_train,    X_test,    y_train, y_test)
evaluate_model("Model 2: Reduced MLR",      LinearRegression(), X_train[top_feats], X_test[top_feats], y_train, y_test)
evaluate_model("Model 3: Standardized MLR", LinearRegression(), X_train_sc, X_test_sc, y_train, y_test)

Model 1: Full MLR                   | R²=0.8471 | RMSE=1428.44 | MAE=950.80
Model 2: Reduced MLR                | R²=0.8415 | RMSE=1454.47 | MAE=968.54
Model 3: Standardized MLR           | R²=0.8471 | RMSE=1428.44 | MAE=950.80


(0.8470757900790584, np.float64(1428.4381127878962), 950.8001606819385)

In [11]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

lasso = Lasso(alpha=10.0)
lasso.fit(X_train_sc, y_train)
y_pred_lasso = lasso.predict(X_test_sc)

print(f"Ridge → R²: {r2_score(y_test, y_pred_ridge):.4f}")
print(f"Lasso → R²: {r2_score(y_test, y_pred_lasso):.4f}")

Ridge → R²: 0.8469
Lasso → R²: 0.8420


## Interview Question 1

**Q1: What is Normalization and Standardization and how is it helpful?**

**Normalization:**
Normalization is a technique used to rescale the values of a feature to a fixed range, usually between 0 and 1.

Formula:
X_norm = (X - X_min) / (X_max - X_min)

When to use:
Use Normalization when the data has no outliers and when you need all values within a bounded range.
Example: Image pixel values are normalized between 0 and 1.

**Standardization:**
Standardization is a technique used to rescale the values of a feature so that the mean becomes 0 and the standard deviation becomes 1.

Formula:
X_std = (X - mean) / standard deviation

When to use:
Use Standardization when the algorithm is sensitive to the scale of features.
Example: Ridge Regression, Lasso Regression, SVM, and Gradient Descent all work better with standardized data.

**How it is helpful in this dataset:**
In the Toyota Corolla dataset, the feature KM ranges from 1 to 243,000 and the feature Doors ranges from 2 to 5.
These two features are on completely different scales.
If we apply Ridge or Lasso without scaling, the model will penalize KM much more than Doors just because of its larger numeric range, not because it is actually more important.
Standardization fixes this problem by putting all features on the same scale before applying the penalty.

**Assumption:**
We assume that the scaler is fit only on the training data and then applied to the test data.
This is done to avoid data leakage, which means the test data should have no influence on the scaling parameters.

## Interview Question 2
Multicollinearity occurs when two or more independent variables in a regression model are highly correlated with each other. For example, in the Toyota Corolla dataset, the variables 'cc' and 'Weight' are both related to engine size, making them correlated.

**Techniques to handle Multicollinearity:**

**1. Variance Inflation Factor (VIF)**
VIF measures how much the variance of a regression coefficient is inflated due to multicollinearity.
A VIF value greater than 10 indicates severe multicollinearity.
Solution: Remove or combine the variable with high VIF.

**2. Ridge Regression (L2 Regularization)**
Ridge adds a penalty equal to alpha multiplied by the sum of squared coefficients to the loss function.
This shrinks all coefficients towards zero but does not eliminate any.
It distributes the effect across correlated features instead of assigning it to just one.

**3. Lasso Regression (L1 Regularization)**
Lasso adds a penalty equal to alpha multiplied by the sum of absolute values of coefficients.
This can shrink some coefficients to exactly zero, effectively removing them from the model.
It automatically selects one variable from a group of correlated variables.

**4. Principal Component Analysis (PCA)**
PCA transforms the original correlated features into a new set of uncorrelated components called principal components.
These components are then used as inputs for regression.
This completely removes multicollinearity but makes the model harder to interpret.

**5. Dropping One Variable**
If two variables are highly correlated, simply remove the less important or less interpretable one.
For example, in this dataset we can drop 'cc' and keep 'Weight' since Weight is more directly interpretable.

**Example from this dataset:**
The correlation between 'cc' and 'Weight' is approximately 0.61, which indicates moderate multicollinearity.
The recommended approach is to use Ridge Regression so that both features are kept in the model while their coefficients are properly controlled.

## Assumptions Made During the Analysis

1. The column 'Cylinders' was dropped because all 1436 rows had the same value (4).
   A constant column adds no information to the model.

2. The value cc = 16000 was treated as a data entry error and corrected to 1600.
   This is because all other cc values are between 1300 and 2000, making 16000 an obvious typo.

3. CNG fuel type was used as the reference category during one-hot encoding.
   This means the coefficients for Diesel and Petrol are interpreted relative to CNG.

4. The StandardScaler was fit only on the training set and then applied to the test set.
   This avoids data leakage and gives a realistic evaluation of model performance.

5. We assumed a linear relationship between the features and Price.
   If the true relationship is non-linear, the MLR model will underperform.

6. We assumed that the features are independent of each other.
   However, cc and Weight show moderate correlation (r = 0.61), which is a mild violation.
   Ridge Regression was used to handle this.